### code

In [33]:

import cv2
import numpy as np

from matplotlib import pyplot as plt
import time

In [34]:
# display helper function
def show_frame(img, title=""):
    plt.figure(figsize=(12, 8))
    if len(img.shape) == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

In [35]:
#parameters tunning helper function
def show_hsv_heatmaps(rgb_image, title="HSV Heatmaps"):
    hsv = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2HSV)

    h, s, v = cv2.split(hsv)

    fig, axes = plt.subplots(1, 4, figsize=(20,5))
    fig.suptitle(title, fontsize=16)

    axes[0].imshow(h, cmap='hot')
    axes[0].set_title("Hue (H)")
    axes[0].axis('off')

    axes[1].imshow(s, cmap='hot')
    axes[1].set_title("Saturation (S)")
    axes[1].axis('off')

    axes[2].imshow(v, cmap='hot')
    axes[2].set_title("Value (V)")
    axes[2].axis('off')

    hsv_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    axes[3].imshow(hsv_rgb)
    axes[3].set_title("HSV image")
    axes[3].axis('off')

    plt.show()


In [36]:
# board detection and board detection helper
def order_points(pts):
    pts = pts.reshape(4, 2)

    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)

    tl = pts[np.argmin(s)]
    br = pts[np.argmax(s)]
    tr = pts[np.argmin(diff)]
    bl = pts[np.argmax(diff)]

    return np.array([tl, tr, br, bl], dtype=np.float32)


def detect_board(frame, min_area_ratio=0.3, max_area_ratio=0.44,
                 BOARD_W=800, BOARD_H=800):

    h, w = frame.shape[:2]
    frame_area = h * w

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    lower_white = np.array([0, 0, 135])
    upper_white = np.array([180, 165, 255])

    white_mask = cv2.inRange(hsv, lower_white, upper_white)
    white_mask = cv2.GaussianBlur(white_mask, (5, 5), 0)

    kernel = np.ones((9,9), np.uint8)
    #show_frame(white_mask,"hsv filtering")
    white_mask = cv2.dilate(white_mask, kernel, iterations=6)
    white_mask[:,-1]=0
    white_mask = cv2.erode(white_mask, kernel, iterations=6)
    #show_frame(white_mask,"morthological operations")

    edges = cv2.Canny(white_mask, 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best_cnt = None
    best_area = 0

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if not (min_area_ratio * frame_area <= area <= max_area_ratio * frame_area):
            continue

        approx = cv2.approxPolyDP(cnt, 0.1 * cv2.arcLength(cnt, True), True)
        if len(approx) == 4 and area > best_area:
            best_cnt = approx
            best_area = area

    if best_cnt is None:
        return None, None, None, None

    src = order_points(best_cnt)

    dst = np.array([
        [0, 0],
        [BOARD_W - 1, 0],
        [BOARD_W - 1, BOARD_H - 1],
        [0, BOARD_H - 1]
    ], dtype=np.float32)

    M = cv2.getPerspectiveTransform(src, dst)
    Minv = cv2.getPerspectiveTransform(dst, src)

    board_warped = cv2.warpPerspective(frame, M, (BOARD_W, BOARD_H))
    #show_frame(board_warped, "upright board")

    return board_warped, M, Minv, src

In [37]:
#red pawns detection
def detect_pawns_verbose(mask, hsv_img, board_w, board_h,
                         min_area_ratio=0.0001, max_area_ratio=0.01,
                         circularity_thresh=0.75, min_saturation=50, min_value=50):
    pawns = []
    board_area = board_w * board_h
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for c in contours:
        area = cv2.contourArea(c)
        peri = cv2.arcLength(c, True)
        circularity = 0 if peri == 0 else 4 * np.pi * area / (peri * peri)
        x, y, w, h = cv2.boundingRect(c)
        roi = hsv_img[y:y+h, x:x+w] if h>0 and w>0 else None
        mean_sat = np.mean(roi[:, :, 1]) if roi is not None else 0
        mean_val = np.mean(roi[:, :, 2]) if roi is not None else 0
        if (min_area_ratio*board_area <= area <= max_area_ratio*board_area and
            circularity >= circularity_thresh and
            mean_sat >= min_saturation and
            mean_val >= min_value):
            pawns.append((x, y, w, h))
    return pawns, None, None

def detect_red_pawns(board, min_sat=190, min_val=130):
    hsv = cv2.cvtColor(board, cv2.COLOR_BGR2HSV)
    lower_red1 = np.array([0,165,160])
    upper_red1 = np.array([10,255,255])
    lower_red2 = np.array([170,165,160])
    upper_red2 = np.array([180,255,255])
    mask_red = cv2.bitwise_or(cv2.inRange(hsv, lower_red1, upper_red1),
                               cv2.inRange(hsv, lower_red2, upper_red2))
    kernel = np.ones((3,3), np.uint8)
    #show_frame(mask_red,"hsv_filtering")
    mask_red = cv2.erode(mask_red, kernel, iterations=2)
    mask_red = cv2.dilate(mask_red, kernel, iterations=5)
    #show_frame(mask_red,"morthological operations")

    overlap_mask=mask_red
    merged = np.zeros_like(board)
    merged[overlap_mask>0] = board[overlap_mask>0]
    merged_hsv = cv2.cvtColor(merged, cv2.COLOR_BGR2HSV)
    sat_mask = merged_hsv[:,:,1] >= min_sat
    val_mask = merged_hsv[:,:,2] >= min_val
    sv_mask = np.logical_and(sat_mask, val_mask).astype(np.uint8)*255
    bright_red_mask = cv2.bitwise_and(mask_red, sv_mask)
    bright_red_mask=mask_red

    kernel = np.ones((3,3), np.uint8)
    red_pawns, _, _ = detect_pawns_verbose(bright_red_mask, merged_hsv, board.shape[1], board.shape[0])
    return red_pawns, bright_red_mask


In [38]:
#blue pawns detection
def detect_pawns(mask, hsv_img, board_w, board_h, min_area_ratio=0.0005, max_area_ratio=0.01, circ_thresh=0.3):
    pawns = []
    board_area = board_w * board_h
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area_ratio*board_area or area > max_area_ratio*board_area:
            continue
        peri = cv2.arcLength(c, True)
        if peri == 0: continue
        circularity = 4*np.pi*area/(peri*peri)
        if circularity < circ_thresh: continue
        x,y,w,h = cv2.boundingRect(c)
        pawns.append((x,y,w,h))
    return pawns

def detect_blue_pawns(board):
    hsv = cv2.cvtColor(board, cv2.COLOR_BGR2HSV)
    lower_blue = np.array([100, 200, 70])
    upper_blue = np.array([130, 255, 255])
    blue_mask = cv2.inRange(hsv, lower_blue, upper_blue)
    #show_frame(blue_mask,"hsv filtering")
    blue_mask = cv2.erode(blue_mask, np.ones((3,3), np.uint8), iterations=9)
    blue_mask = cv2.dilate(blue_mask, np.ones((3,3), np.uint8), iterations=7)
    #show_frame(blue_mask,"morthological opererations")
    blue_pawns = detect_pawns(blue_mask, hsv, board.shape[1], board.shape[0])
    return blue_pawns, blue_mask

In [39]:
# die detection and helpers
def extract_rotated_roi(image, rect):
    center, (w, h), angle = rect

    if w < h:
        w, h = h, w
        angle += 90

    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))

    x = int(center[0] - w/2)
    y = int(center[1] - h/2)
    w = int(w)
    h = int(h)

    return rotated[y:y+h, x:x+w]

def get_upright_die(frame, rect, shrink=0.85):
    print(rect,"rect_content")
    (cx, cy), (w, h), angle = rect

    if w < h:
        w, h = h, w
        angle += 90

    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    rotated = cv2.warpAffine(frame, M, (frame.shape[1], frame.shape[0]))

    w = int(w * shrink)
    h = int(h * shrink)

    x = int(cx - w / 2)
    y = int(cy - h / 2)

    return rotated[y:y+h, x:x+w]




def detect_dice(frame, board_bbox):
    x1, y1, w1, h1 = board_bbox
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower_white = np.array([10, 0, 180])
    upper_white = np.array([170, 60, 255])

    white_mask = cv2.inRange(hsv, lower_white, upper_white)


    gray_full = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_full = cv2.bitwise_and(gray_full, gray_full, mask=white_mask)

    _, dice_mask = cv2.threshold(gray_full, 120, 255, cv2.THRESH_BINARY)
    dice_mask[y1:y1+h1, x1:x1+w1] = 0
    #show_frame(dice_mask,"hsv filtering")


    contours, _ = cv2.findContours(dice_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    dice_rects = []
    frame_area = frame.shape[0]*frame.shape[1]
    dice_id=0
    h_frame, w_frame = frame.shape[:2]
    dice_coords=[]
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if not (0.005*frame_area >= area >= 0.0005*frame_area):
            continue
        rect = cv2.minAreaRect(cnt) 
        (box_w, box_h) = rect[1]
    
        if min(box_w, box_h) == 0:
            continue

        aspect_ratio = max(box_w, box_h) / min(box_w, box_h)
        if aspect_ratio > 2.5: 
            continue

        die_roi = extract_rotated_roi(frame, rect)

        if die_roi is None or die_roi.size == 0:
            continue
    
        dice_rects.append(die_roi)
        cx, cy = rect[0]
        w, h = rect[1]
        x = int(cx - w/2)
        y = int(cy - h/2)
        dice_coords.append((x, y, int(w), int(h)))
        
    dice_numbers = []

    for rect in dice_rects:

        if rect is None:
            dice_numbers.append(0)
            continue
        #show_frame(rect,"upright dice")
        gray = cv2.cvtColor(rect, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)

        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

        kernel = np.ones((3, 3), np.uint8)
        thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
        #show_frame(thresh,"hsv filtering")
        h, w = thresh.shape
        border = int(0.12 * min(h, w))
        thresh[:border, :] = 0
        thresh[-border:, :] = 0
        thresh[:, :border] = 0
        thresh[:, -border:] = 0

        contours, _ = cv2.findContours(
            thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )

        dot_count = 0
        die_area = h * w

        for c in contours:
            area = cv2.contourArea(c)
            if area <= 0:
                continue

            peri = cv2.arcLength(c, True)
            if peri == 0:
                continue

            circularity = 4 * np.pi * area / (peri * peri)

            if (
                0.004 * die_area < area < 0.035 * die_area and
                circularity > 0.75
            ):
                dot_count += 1

        dice_numbers.append(dot_count)

    return dice_coords, dice_numbers

In [40]:
#card detection
def detect_card(frame,board_corners=None):
    frame_new=frame.copy()
    if board_corners is not None:
        ll_corner_x = int(board_corners[3, 0])
        ll_corner_y = int(board_corners[3, 1])

        frame_new[:, ll_corner_x:] = 0 

    hsv_full = cv2.cvtColor(frame_new, cv2.COLOR_BGR2HSV)
    lower_blue = np.array([80,80,40])
    upper_blue = np.array([145,255,255])

    lower_red1 = np.array([0,140,40])
    upper_red1 = np.array([10,255,220])

    lower_red2 = np.array([170,140,40])
    upper_red2 = np.array([179,255,220])

    mask_blue = cv2.inRange(hsv_full, lower_blue, upper_blue)
    mask_red = cv2.inRange(hsv_full, lower_red1, upper_red1) | \
            cv2.inRange(hsv_full, lower_red2, upper_red2)

    mask = mask_blue | mask_red
    #show_frame(mask,"color filtering")

    sat_mask = cv2.bitwise_and(frame, frame, mask=mask)
    sat_mask_hsv = cv2.cvtColor(sat_mask, cv2.COLOR_BGR2HSV)
    sat_mask = cv2.inRange(sat_mask_hsv, (0,80,50), (180,255,255))
    sat_mask = cv2.erode(sat_mask, np.ones((5,5), np.uint8), iterations=2)
    contours, _ = cv2.findContours(sat_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    frame_area = frame.shape[0] * frame.shape[1]
    #show_frame(sat_mask,"saturation thresholding")

    large_contours = [cnt for cnt in contours if cv2.contourArea(cnt) >= 0.01*frame_area]

    if not large_contours:
        return None, None, None

    biggest_contour = max(large_contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(biggest_contour)
    roi = frame[y:y+h, x:x+w]
    b, g, r = cv2.mean(roi)[:3]

    if r > g and r > b:
        color_name, color_val = "Red", (0, 0, 255)
    elif b > r and b > g:
        color_name, color_val = "Blue", (255, 0, 0)
    else:
        color_name, color_val = "Unknown", (255, 255, 255)

    return (x, y, w, h), color_name, color_val

In [41]:

# full frame and events and status processing along with helpers
dice_prev_rel = None
last_die_value=None
last_die_stable_frames=0 
card_prev_state = None
card_stable_frames = 0
card_last_none_frames = 0
no_none_to_save=0
CARD_STABLE_THRESHOLD = 20  
MESSAGE_DURATION = 80 
board_prev_center = None    
board_stable_frames = 0       
board_moving_frames = 0       
board_message_shown = False   
BOARD_STABLE_THRESHOLD = 20 

def draw_board_rect_on_frame(frame, Minv, x, y, w, h, color, thickness=2):
    pts = np.array([
        [x, y],
        [x + w, y],
        [x + w, y + h],
        [x, y + h]
    ], dtype=np.float32).reshape(-1, 1, 2)

    pts_frame = cv2.perspectiveTransform(pts, Minv)
    cv2.polylines(frame, [pts_frame.astype(int)], True, color, thickness)

def put_text_on_board(final, Minv, text, pos, font_scale=0.55, color=(0, 255, 0), thickness=2, font=cv2.FONT_HERSHEY_SIMPLEX):
    board_point = np.array([[pos]], dtype=np.float32)
    frame_point = cv2.perspectiveTransform(board_point, Minv)
    x, y = frame_point[0][0]
    cv2.putText(
        final,
        text,
        (int(x), int(y)),
        font,
        font_scale,
        color,
        thickness,
        cv2.LINE_AA
    )


def process_frame(frame,  previous_board = (None, None, None, None)):
    if not hasattr(process_frame, "previous_gameplay_score"):
        process_frame.previous_gameplay_score = None

    if not hasattr(process_frame,"change_candidates"):
        process_frame.change_candidates= {}

    if not hasattr(process_frame, "state"):
        process_frame.state = {}

    if not hasattr(process_frame, "active_messages"):
        process_frame.active_messages = []
    global dice_prev_rel, board_prev_center, board_stable_frames, board_moving_frames, board_message_shown, BOARD_STABLE_THRESHOLD   
    global card_prev_state, card_stable_frames, card_last_none_frames, no_none_to_save, MESSAGE_DURATION
    global last_die_value, last_die_stable_frames 
    board, M, Minv, board_corners = detect_board(frame)
    if board is None:
        if previous_board[0] is not None:
            _, prev_M, prev_Minv, prev_corners = previous_board
            
            board_size = (800, 800) 
            board = cv2.warpPerspective(frame, prev_M, board_size)
            M = prev_M
            Minv = prev_Minv
            board_corners = prev_corners
        else:
            return frame, previous_board  

    red_pawns, _ = detect_red_pawns(board)
    blue_pawns, _ = detect_blue_pawns(board) 
    h, w = board.shape[:2]
    board_outline = np.array([
        [[0, 0]],
        [[w-1, 0]],
        [[w-1, h-1]],
        [[0, h-1]]
    ], dtype=np.float32)
    outline_frame = cv2.perspectiveTransform(board_outline, Minv) 
   
    outline_pts = outline_frame[:, 0, :]  
    x_min = int(np.min(outline_pts[:, 0]))
    y_min = int(np.min(outline_pts[:, 1]))
    x_max = int(np.max(outline_pts[:, 0]))
    y_max = int(np.max(outline_pts[:, 1]))
    board_bbox = (x_min, y_min, x_max - x_min, y_max - y_min)

    dice_rects, dice_numbers = detect_dice(frame, board_bbox)
    card_box, card_color, card_draw = detect_card(frame,board_corners)

    final = frame.copy()
    board_contour = board_corners.astype(int)
    if board_contour.ndim == 2:  
        board_contour = board_contour[:, None, :]
    x, y, w, h = cv2.boundingRect(board_contour)
    board_bbox = (x, y, w, h)


    cv2.drawContours(final, [board_contour], -1, (0,255,0), 10)
    x1, y1, w, h = board_bbox
    top_left = outline_frame[0][0] 
    cv2.putText(final, "Board",
                (int(top_left[0]), int(top_left[1]) - 20), 
                cv2.FONT_HERSHEY_SIMPLEX,
                1.5,
                (0,255,0),
                4)
    img_h, img_w = final.shape[:2]
    x, y, w, h = board_bbox
    board_center_x = int(np.mean(outline_pts[:, 0]))
    board_center_y = int(np.mean(outline_pts[:, 1]))
    board_center = (board_center_x, board_center_y)

    die_value=0
    for (x, y, w, h), num in zip(dice_rects, dice_numbers):
        cv2.rectangle(final, (x, y), (x+w, y+h), (0,0,255), 3)
        cv2.putText(final, f"Die:{num}", (x, y-15),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,0,255), 3)
        die_value=num
        
    img_h, img_w = final.shape[:2]

    dice_coords = []
    for (x, y, w, h), num in zip(dice_rects, dice_numbers):
        cx = round((x + w/2) / img_w, 3)
        cy = round((y + h/2) / img_h, 3)
        dice_coords.append({
            "value": num,
            "cx": cx,
            "cy": cy
        })

    MOVE_THRESHOLD = 100  
    if dice_coords:
        dice_cx = dice_coords[0]["cx"] 
        dice_cy = dice_coords[0]["cy"]
        rel = (
            (dice_cx - board_center[0])*10000,
            (dice_cy - board_center[1])*10000
        )
    else:
        rel = None

    if dice_prev_rel is not None and rel is not None:
        dist=(rel[0]**2+rel[1]**2)**0.5-(dice_prev_rel[0]**2+dice_prev_rel[1]**2)**0.5
        if die_value == last_die_value:
            last_die_stable_frames += 1
        else:
            last_die_stable_frames = 0
            last_die_value = die_value

        if dist < MOVE_THRESHOLD and dist>-MOVE_THRESHOLD:
            board_stable_frames += 1
            board_moving_frames = max(0, board_moving_frames) 
            if board_stable_frames>BOARD_STABLE_THRESHOLD-10:
                board_moving_frames=0
        else:
            board_stable_frames = 0
            board_moving_frames += 1
    else:
        board_stable_frames = 0
        board_moving_frames = 0

    dice_prev_rel = rel

    if last_die_stable_frames == BOARD_STABLE_THRESHOLD and  board_stable_frames >= 5 :
        if not board_message_shown:
            msg_text = f"The dice has rolled: {die_value}"
            msg_color = (0, 255, 0) 
            expire_time = time.time() + 5 
            process_frame.active_messages.append([msg_text, msg_color, expire_time])
            board_stable_frames=0

    if card_box:
        x, y, w, h = card_box
        cv2.rectangle(final, (x, y), (x+w, y+h), card_draw, 4)
        cv2.putText(final, card_color, (x, y-20),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.4, card_draw, 4)
    current_turn=card_color
            
    CARD_STABLE_THRESHOLD = 10

    if card_color == card_prev_state:
        card_stable_frames += 1
    else:
        card_stable_frames = 1
        card_prev_state = card_color

    if card_color is None:
        card_last_none_frames += 1
    else:
        if card_last_none_frames>0:
            no_none_to_save=card_last_none_frames
        card_last_none_frames = 0

    if card_color is not None:
        if card_stable_frames == CARD_STABLE_THRESHOLD and no_none_to_save >= CARD_STABLE_THRESHOLD:
            card_message = f"Now is the: {card_color.upper()} turn"
            card_color_to_display = (0,0,255) if card_color.lower() == "red" else (255,0,0)
            expire_time = time.time() + 5

            if not any(m[0] == card_message for m in process_frame.active_messages):
                process_frame.active_messages.append([card_message, card_color_to_display, expire_time])

    for x,y,w,h in red_pawns:
        draw_board_rect_on_frame(final, Minv, x, y, w, h, (0,0,255), 2)
    for x,y,w,h in blue_pawns:
        draw_board_rect_on_frame(final, Minv, x, y, w, h, (255,0,0), 2)

    board_h, board_w = board.shape[:2]

    red_pawn_coords = []
    for x, y, w, h in red_pawns:
        cx = round((x + w/2) / board_w, 2)
        cy = round((y + h/2) / board_h, 2)
        red_pawn_coords.append((cx, cy))

    blue_pawn_coords = []
    for x, y, w, h in blue_pawns:
        cx = round((x + w/2) / board_w, 2)
        cy = round((y + h/2) / board_h, 2)
        blue_pawn_coords.append((cx, cy))

    ZONES = {
        "blue_base": {
            "x": 0.0, "y": 0.8, "w": 0.20, "h": 0.2
        },
        "blue_home": {
            "x": 0.17, "y": 0.43, "w": 0.27, "h": 0.14
        },
        "red_base": {
            "x": 0.0, "y": 0.0, "w": 0.2, "h": 0.2
        },
        "red_home": {
            "x": 0.42, "y": 0.19, "w": 0.14, "h": 0.27
        }
    }

    def zone_to_pixels(zone, board_w, board_h):
        x = int(zone["x"] * board_w)
        y = int(zone["y"] * board_h)
        w = int(zone["w"] * board_w)
        h = int(zone["h"] * board_h)
        return x, y, w, h
    

    for name, zone in ZONES.items():
        zx, zy, zw, zh = zone_to_pixels(zone, board_w, board_h)
        color = (255,0,0) if "blue" in name else (0,0,255)

        draw_board_rect_on_frame(final, Minv, zx, zy, zw, zh, color,3)

        label = name.replace("_", " ").title()

        text_x = zx + 6
        text_y = zy + 20


        put_text_on_board(final, Minv, 
            label,
            (text_x, text_y),
            0.55,
            color,
            2
        )

    
    def point_in_zone(cx, cy, zone):
        return (
            zone["x"] <= cx <= zone["x"] + zone["w"] and
            zone["y"] <= cy <= zone["y"] + zone["h"]
        )
    
    def classify_pawn(cx, cy):
        for name, zone in ZONES.items():
            if point_in_zone(cx, cy, zone):
                return name
        return "board"
    
    ZONE_COLORS = {
        "blue_base": (255,0,0),
        "blue_home": (255,150,150),
        "red_base": (0,0,255),
        "red_home": (150,150,255),
        "board": (200,200,200)
    }

    for (x,y,w,h), (cx,cy) in zip(red_pawns, red_pawn_coords):
        zone = classify_pawn(cx, cy)
        draw_board_rect_on_frame(final, Minv, x, y, w, h, ZONE_COLORS[zone], 2)
        label = zone
        text_x = x
        text_y = max(y - 6, 12)

        put_text_on_board(final, Minv, 
            label,
            (text_x, text_y),
            0.4,
            (0,0,255),
            1
        )

    for (x,y,w,h), (cx,cy) in zip(blue_pawns, blue_pawn_coords):
        zone = classify_pawn(cx, cy)
        draw_board_rect_on_frame(final, Minv, x, y, w, h, ZONE_COLORS[zone], 2)
        label =zone

        text_x = x
        text_y = max(y - 6, 12)

        put_text_on_board(final, Minv, 
            label,
            (text_x, text_y),
            0.4,
            (255,0,0),
            1
        )

    board_h, board_w = board_bbox[3], board_bbox[2]  
    board_resized = cv2.resize(board, (board_w, board_h))

    def compute_gameplay_score(red_coords, blue_coords):
        score = {
            "red_base":0, "red_home":0, "red_board":0,
            "blue_base":0, "blue_home":0, "blue_board":0
        }
        for cx, cy in red_coords:
            zone = classify_pawn(cx, cy)
            key = zone if "red" in zone else "red_board"
            score[key] += 1
        for cx, cy in blue_coords:
            zone = classify_pawn(cx, cy)
            key = zone if "blue" in zone else "blue_board"
            score[key] += 1
        return score

    if not hasattr(process_frame, "previous_gameplay_score"):
        process_frame.previous_gameplay_score = None

    if len(red_pawn_coords) == 4 and len(blue_pawn_coords) == 4:
        gameplay_score = compute_gameplay_score(red_pawn_coords, blue_pawn_coords)

        gameplay_changes = {}
        if process_frame.previous_gameplay_score is not None:
            for key in gameplay_score.keys():
                prev = process_frame.previous_gameplay_score.get(key, 0)
                curr = gameplay_score.get(key, 0)
                if curr != prev:
                    gameplay_changes[key] = f"{curr - prev:+}"
                    process_frame.state=gameplay_score.copy()

        process_frame.previous_gameplay_score = gameplay_score
    else:
        gameplay_score = process_frame.previous_gameplay_score
        gameplay_changes = {} 

    
    if gameplay_score is not None:
        panel_w, panel_h = 250, 380
        panel_x, panel_y = 5, frame.shape[0] - panel_h - 10
        overlay = frame.copy()
        cv2.rectangle(final, (panel_x, panel_y), (panel_x+panel_w, panel_y+panel_h), (50,50,50), -1)
        cv2.addWeighted(final, 0.2, frame, 0.8, 0, frame)
        line_height = 22
        start_y = panel_y + 25
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(final, "Game Status", (panel_x + 5, panel_y + 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 2)

        cv2.putText(final, f"Red Base: {gameplay_score.get('red_base', 0)}", (panel_x + 10, start_y), font, 0.5, (0,0,255), 1)
        cv2.putText(final, f"Red Home: {gameplay_score.get('red_home', 0)}", (panel_x + 10, start_y + line_height), font, 0.5, (0,0,255), 1)
        cv2.putText(final, f"Red Board: {gameplay_score.get('red_board', 0)}", (panel_x + 10, start_y + 2*line_height), font, 0.5, (0,0,255), 1)
        line_offset = 0
        for idx, coord in enumerate(red_pawn_coords):
            cv2.putText(final, f"Red Pawn {idx+1}: {coord}", 
                        (panel_x + 10, start_y + 3*line_height + line_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
            line_offset += line_height  
        
        cv2.putText(final, f"Blue Base: {gameplay_score.get('blue_base', 0)}", (panel_x + 10, start_y + 7*line_height), font, 0.5, (255,0,0), 1)
        cv2.putText(final, f"Blue Home: {gameplay_score.get('blue_home', 0)}", (panel_x + 10, start_y + 8*line_height), font, 0.5, (255,0,0), 1)
        cv2.putText(final, f"Blue Board: {gameplay_score.get('blue_board', 0)}", (panel_x + 10, start_y + 9*line_height), font, 0.5, (255,0,0), 1)
        line_offset = 0
        for idx, coord in enumerate(blue_pawn_coords):
            cv2.putText(final, f"Blue Pawn {idx+1}: {coord}", 
                        (panel_x + 10, start_y + 10*line_height + line_offset),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1)
            line_offset += line_height

        cv2.putText(final,  f"Turn: {current_turn}", (panel_x + 10, start_y + 14*line_height), font, 0.5, (0,255,0), 1)

        cv2.putText(final, f"Die: {die_value}", (panel_x + 10, start_y + 15*line_height), font, 0.5, (0,255,255), 1)

        red_home = gameplay_score.get('red_home', 0)
        blue_home = gameplay_score.get('blue_home', 0)
        if red_home > blue_home:
            winner = "Red"
        elif blue_home > red_home:
            winner = "Blue"
        else:
            winner = "Draw"

        cv2.putText(final, f"Gameplay Score: Winner - {winner}", (panel_x + 10, start_y + 16*line_height), font, 0.5, (255,255,255), 1)
        
    STABLE_FRAMES = 10 
    MESSAGE_DURATION = 5.0 

    current_time = time.time()

    if not hasattr(process_frame, "pending_changes"):
        process_frame.pending_changes = {}

    if not hasattr(process_frame, "global_stable_frames"):
        process_frame.global_stable_frames = 0

    if gameplay_changes:
        process_frame.global_stable_frames = 0  

        for key, delta_str in gameplay_changes.items():
            process_frame.pending_changes[key] = int(delta_str)

    else:
        process_frame.global_stable_frames += 1

    if process_frame.global_stable_frames >= STABLE_FRAMES:

        for key, delta in process_frame.pending_changes.items():

            if key.endswith("base") and delta < 0:
                text = "Pawn left base"
            elif key.endswith("base") and delta > 0:
                text = "Pawn returned to base"
            elif key.endswith("home") and delta > 0:
                text = "Pawn entered home"
            else:
                continue

            color = (0,0,255) if "red" in key else (255,0,0)
            if "home" in key:
                color = (150,150,255) if "red" in key else (255,150,150)

            expire_time = current_time + MESSAGE_DURATION

            if not any(m[0] == text for m in process_frame.active_messages):
                process_frame.active_messages.append((text, color, expire_time))

        process_frame.pending_changes.clear()
        process_frame.global_stable_frames = 0


    line_height=22
    msg_x, msg_y = 10, 50
    new_active_messages = []
    for text, color, expire_time in process_frame.active_messages:
        if current_time <= expire_time:
            cv2.putText(final, text, (msg_x, msg_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
            msg_y += line_height
            new_active_messages.append((text, color, expire_time))

    process_frame.active_messages = new_active_messages

    

    return final, (board, M, Minv, board_corners) 



In [42]:
#loop to check/generate a snippet of a video

input_video = "easy_2.mp4"
output_video = "experiment.mp4"

start_time = 10  
end_time   = 20  


cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise FileNotFoundError("Could not open input video")

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))


start_frame = int(start_time * fps)
end_frame   = int(end_time * fps)

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

previous_board = (None,None,None,None)
current_frame = start_frame

while cap.isOpened() and current_frame <= end_frame:
    ret, frame = cap.read()
    if not ret:
        break

    result, contour = process_frame(frame, previous_board)
    previous_board = contour
    out.write(result)

    current_frame += 1

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 

In [ ]:

#loop to generate entire video
input_video = "dif_2.mp4"
output_video = "placeholder.mp4"


cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise FileNotFoundError("Could not open input video")

fps = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    output_video, fourcc, fps, (width, height)
)

previous_board = (None,None,None,None)  

while True:
    ret, frame = cap.read()
    if not ret:
        break

    result,contour = process_frame(frame,previous_board)
    previous_board=contour
    out.write(result)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
